In [ ]:
# Find the pipeline package without walking the DICOM tree.
import sys, os, time, gc, json
from pathlib import Path

T0 = time.time()
def log(msg):
    print(f"[{time.time() - T0:7.1f}s] {msg}", flush=True)

def walk(root="/kaggle/input", max_depth=8, suffixes=None):
    base = len(Path(root).parts)
    for dirpath, dirnames, filenames in os.walk(root):
        here = Path(dirpath)
        if len(here.parts) - base >= max_depth:
            dirnames[:] = []
        else:
            dirnames[:] = [d for d in dirnames if not d.lower().endswith(("_series", "_images"))]
        for f in filenames:
            if suffixes is None or Path(f).suffix.lower() in suffixes:
                yield here / f

PKG = next((p.parent.parent for p in walk(suffixes=(".py",))
            if p.name == "targets.py" and p.parent.name == "rsna_knee"), None)
if PKG is None:
    raise FileNotFoundError("attach the rsna-knee-pipeline dataset (v2, it must contain rsna_knee/targets.py)")
sys.path.insert(0, str(PKG))

import numpy as np
import pandas as pd
from rsna_knee.report_labels import TARGETS
log(f"pipeline: {PKG}")


In [ ]:
from rsna_knee.infer import find_root, write_benchmark, write_submission, validate_submission
ROOT = find_root()
test = pd.read_csv(ROOT / "test.csv")
SUB = Path("/kaggle/working/submission.csv")

# family weights for the rank blend; families not listed get 1.0
FAMILY_WEIGHTS = {}
# False: no constant file is written first, and any inference failure stops the notebook,
# so a timeout or a crash leaves no submission.csv instead of a benchmark that looks valid
ALLOW_BENCHMARK = False
if ALLOW_BENCHMARK:
    write_benchmark(test, SUB)
    log(f"benchmark written for {len(test)} studies")
# every mounted family must hold fold0 ... fold4; set None to allow a single-fold family
REQUIRE_FOLDS = 5


In [ ]:
from rsna_knee.data import CacheConfig, build_cache
from rsna_knee.infer2 import find_checkpoints, check_folds, predict_family, rank_blend, family_cache
from rsna_knee.ntejas_arms import find_arm_checkpoints, predict_arm

FAMILIES = find_checkpoints(walk)
if REQUIRE_FOLDS:
    check_folds(FAMILIES, REQUIRE_FOLDS)
ARMS = find_arm_checkpoints(walk)
NEED = {}
for k, v in FAMILIES.items():
    NEED[k] = family_cache(v)
    log(f"family {k} ({NEED[k]}): {[p.name for p in v]}")
for k, v in ARMS.items():
    log(f"public arm {k}: {[p.name for p in v]}")
if not FAMILIES and not ARMS:
    raise FileNotFoundError("no checkpoints mounted")
test_series = pd.read_csv(ROOT / "test_series.csv")


In [ ]:
try:
    members = {}
    if "v1" in NEED.values():
        CFG = CacheConfig(px=288, slices_per_slot=12, group=3, crop_mm=130.0)
        C1 = Path("/kaggle/temp/test_cache"); C1.mkdir(parents=True, exist_ok=True)
        a1, m1, s1 = build_cache(ROOT, test_series, test.StudyInstanceUID.tolist(), "test", CFG, C1, log=log)
        assert s1 == test.StudyInstanceUID.tolist()
        log(f"v1 test cache done at {(time.time() - T0) / 60:.1f} min")
    if "v3" in NEED.values() or ARMS:
        from rsna_knee.cache3 import build_split
        a3, m3, s3 = build_split(ROOT, "test", test.StudyInstanceUID.tolist(), Path("/kaggle/temp/c3"), log=log)
        log(f"v3 test cache done at {(time.time() - T0) / 60:.1f} min")
    for tag, paths in FAMILIES.items():
        arr, msk = (a1, m1) if NEED[tag] == "v1" else (a3, m3)
        members[tag] = predict_family(paths, arr, msk, log=log)
        log(f"family {tag} done at {(time.time() - T0) / 60:.1f} min")
    for arm, paths in ARMS.items():
        members["ntejas_" + arm] = predict_arm(paths, arm, a3, m3, log=log)
        log(f"arm {arm} done at {(time.time() - T0) / 60:.1f} min")
    weights = {k: FAMILY_WEIGHTS.get(k, 0.5 if k.startswith("ntejas_") else 1.0) for k in members}
    log(f"blend weights: {weights}")
    pred = rank_blend(members, weights)
    out = write_submission(pred, test.StudyInstanceUID.tolist(), test, SUB)
    validate_submission(SUB, test)
    log(f"submission written and validated: {out.shape}")
except Exception:
    import traceback
    traceback.print_exc()
    if not ALLOW_BENCHMARK:
        raise
    log("inference failed; the benchmark file stays in place (ALLOW_BENCHMARK = True)")
assert pd.read_csv(SUB)[list(TARGETS)].nunique().min() > 1 or ALLOW_BENCHMARK, "submission is constant"
log(f"total {(time.time() - T0) / 60:.1f} min")
